# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/Users/danie/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [6]:
X = wine.data.astype(np.float32)
y = wine.target.astype(np.int32)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

In [8]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

In [9]:
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes=num_classes)

In [10]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(num_features,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

In [11]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
13/13 [==============================] - 0s 6ms/step - loss: 1.0683 - accuracy: 0.3939 - val_loss: 0.9641 - val_accuracy: 0.4800
Epoch 2/20
13/13 [==============================] - 0s 1ms/step - loss: 0.8021 - accuracy: 0.7778 - val_loss: 0.7486 - val_accuracy: 0.8000
Epoch 3/20
13/13 [==============================] - 0s 1ms/step - loss: 0.6152 - accuracy: 0.9596 - val_loss: 0.5603 - val_accuracy: 0.9600
Epoch 4/20
13/13 [==============================] - 0s 1ms/step - loss: 0.4613 - accuracy: 0.9899 - val_loss: 0.3926 - val_accuracy: 1.0000
Epoch 5/20
13/13 [==============================] - 0s 1ms/step - loss: 0.3328 - accuracy: 0.9899 - val_loss: 0.2767 - val_accuracy: 1.0000
Epoch 6/20
13/13 [==============================] - 0s 2ms/step - loss: 0.2396 - accuracy: 0.9899 - val_loss: 0.1883 - val_accuracy: 1.0000
Epoch 7/20
13/13 [==============================] - 0s 1ms/step - loss: 0.1704 - accuracy: 0.9899 - val_loss: 0.1289 - val_accuracy: 1.0000
Epoch 8/20
13/13 [==

In [12]:
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print(f"Test Accuracy: {test_acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

2/2 [==============================] - 0s 987us/step
Test Accuracy: 0.9815

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       1.00      0.95      0.98        21
           2       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]


In [13]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("model_base.tflite", "wb") as f:
    f.write(tflite_model)

base_size_kb = os.path.getsize("model_base.tflite") / 1024
print(f"Float32 TFLite model size: {base_size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmpo_x232ux/assets


INFO:tensorflow:Assets written to: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmpo_x232ux/assets


Float32 TFLite model size: 14.07 KB


2026-05-19 13:26:24.533748: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-19 13:26:24.533845: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-19 13:26:24.534343: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmpo_x232ux
2026-05-19 13:26:24.534742: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 13:26:24.534747: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmpo_x232ux
2026-05-19 13:26:24.535969: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 optimization pass is not enabled
2026-05-19 13:26:24.536337: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-19 13:26:24.556857: I tensorflow/cc/saved_model/loader.

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [14]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    tflite_model = converter.convert()

    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_index = input_details[0]["index"]
    output_index = output_details[0]["index"]

    y_pred = []

    for i in range(len(X_test)):
        input_data = X_test[i:i + 1].astype(np.float32)

        if input_details[0]["dtype"] in [np.int8, np.uint8]:
            input_scale, input_zero_point = input_details[0]["quantization"]
            input_data = input_data / input_scale + input_zero_point
            input_data = np.round(input_data).astype(input_details[0]["dtype"])
        else:
            input_data = input_data.astype(input_details[0]["dtype"])

        interpreter.set_tensor(input_index, input_data)
        interpreter.invoke()

        output_data = interpreter.get_tensor(output_index)

        if output_details[0]["dtype"] in [np.int8, np.uint8]:
            output_scale, output_zero_point = output_details[0]["quantization"]
            output_data = output_scale * (
                output_data.astype(np.float32) - output_zero_point
            )

        y_pred.append(np.argmax(output_data, axis=1)[0])

    y_pred = np.array(y_pred)
    y_true = np.argmax(y_test_cat, axis=1)

    accuracy = np.mean(y_pred == y_true)

    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")
    print(f"{quant_type.upper()} TFLite accuracy: {accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    return accuracy, file_size_kb(filename)


In [15]:
def file_size_kb(filename):
    """Return file size in KB."""
    return os.path.getsize(filename) / 1024
    
int8_acc, int8_size = quantize_and_evaluate(
    model,
    X_test,
    y_test_cat,
    quant_type='int8',
    filename='model_int8.tflite'
)

float16_acc, float16_size = quantize_and_evaluate(
    model,
    X_test,
    y_test_cat,
    quant_type='float16',
    filename='model_float16.tflite'
)

dynamic_acc, dynamic_size = quantize_and_evaluate(
    model,
    X_test,
    y_test_cat,
    quant_type='dynamic',
    filename='model_dynamic.tflite'
)

INFO:tensorflow:Assets written to: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmp2z7yk5wt/assets


INFO:tensorflow:Assets written to: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmp2z7yk5wt/assets
/Users/danie/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



INT8 TFLite model size: 5.74 KB
INT8 TFLite accuracy: 0.9815

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       1.00      0.95      0.98        21
           2       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]
INFO:tensorflow:Assets written to: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmpwg89ba5z/assets


2026-05-19 13:26:24.963497: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-19 13:26:24.963507: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-19 13:26:24.963620: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmp2z7yk5wt
2026-05-19 13:26:24.963981: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 13:26:24.963985: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmp2z7yk5wt
2026-05-19 13:26:24.965033: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-19 13:26:24.981299: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmp2z7yk5wt
2026-05-


FLOAT16 TFLite model size: 8.95 KB
FLOAT16 TFLite accuracy: 0.9815

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       1.00      0.95      0.98        21
           2       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]
INFO:tensorflow:Assets written to: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmp7oz4ah6_/assets


2026-05-19 13:26:25.244803: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-19 13:26:25.244820: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-19 13:26:25.244932: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmpwg89ba5z
2026-05-19 13:26:25.245334: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 13:26:25.245339: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmpwg89ba5z
2026-05-19 13:26:25.246580: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-19 13:26:25.263943: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmpwg89ba5z
2026-05-


DYNAMIC TFLite model size: 8.17 KB
DYNAMIC TFLite accuracy: 0.9815

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       1.00      0.95      0.98        21
           2       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]


2026-05-19 13:26:25.498864: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmp7oz4ah6_
2026-05-19 13:26:25.499295: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 13:26:25.499299: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmp7oz4ah6_
2026-05-19 13:26:25.500369: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-19 13:26:25.518067: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmp7oz4ah6_
2026-05-19 13:26:25.522607: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 23743 microseconds.


## Problem 1 - Part (c)

### Pruning

In [16]:
batch_size = 8
epochs = 10

end_step = int(np.ceil(len(X_train) / batch_size) * epochs)

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step
)

In [17]:
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

pruned_model = tf.keras.Sequential([
    prune_low_magnitude(
        tf.keras.layers.Dense(
            64,
            activation='relu',
            input_shape=(num_features,)
        ),
        pruning_schedule=pruning_schedule
    ),

    prune_low_magnitude(
        tf.keras.layers.Dense(32, activation='relu'),
        pruning_schedule=pruning_schedule
    ),

    prune_low_magnitude(
        tf.keras.layers.Dense(num_classes, activation='softmax'),
        pruning_schedule=pruning_schedule
    )
])

In [18]:
pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep()
]

history_pruned = pruned_model.fit(
    X_train,
    y_train_cat,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 1s 5ms/step - loss: 1.0131 - accuracy: 0.5253 - val_loss: 0.8849 - val_accuracy: 0.8000
Epoch 2/10
13/13 [==============================] - 0s 1ms/step - loss: 0.7459 - accuracy: 0.8586 - val_loss: 0.6701 - val_accuracy: 0.8800
Epoch 3/10
13/13 [==============================] - 0s 1ms/step - loss: 0.5543 - accuracy: 0.9394 - val_loss: 0.5035 - val_accuracy: 0.9200
Epoch 4/10
13/13 [==============================] - 0s 1ms/step - loss: 0.4030 - accuracy: 0.9596 - val_loss: 0.3727 - val_accuracy: 0.9200
Epoch 5/10
13/13 [==============================] - 0s 1ms/step - loss: 0.2896 - accuracy: 0.9798 - val_loss: 0.2854 - val_accuracy: 0.9200
Epoch 6/10
13/13 [==============================] - 0s 1ms/step - loss: 0.2053 - accuracy: 1.0000 - val_loss: 0.2289 - val_accuracy: 0.9200
Epoch 7/10
13/13 [==============================] - 0s 1ms/step - loss: 0.1501 - accuracy: 1.0000 - val_loss: 0.1959 - val_accuracy: 0.9200
Epoch 8/10
13/13 [==

In [19]:
stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
pruned_tflite_model = converter.convert()

with open("model_pruned.tflite", "wb") as f:
    f.write(pruned_tflite_model)

pruned_size_kb = file_size_kb("model_pruned.tflite")

print(f"Pruned TFLite model size: {pruned_size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmplridgh_2/assets


INFO:tensorflow:Assets written to: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmplridgh_2/assets


Pruned TFLite model size: 14.14 KB


2026-05-19 13:26:26.649851: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-19 13:26:26.649861: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-19 13:26:26.649953: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmplridgh_2
2026-05-19 13:26:26.650218: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 13:26:26.650222: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmplridgh_2
2026-05-19 13:26:26.650823: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-19 13:26:26.657327: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmplridgh_2
2026-05-

In [20]:
stripped_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

test_loss, test_acc = stripped_model.evaluate(
    X_test,
    y_test_cat,
    verbose=0
)

y_pred_probs = stripped_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

y_true = np.argmax(y_test_cat, axis=1)

print(f"Pruned model accuracy: {test_acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

2/2 [==============================] - 0s 763us/step
Pruned model accuracy: 0.9630

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       0.95      0.95      0.95        21
           2       0.93      0.93      0.93        15

    accuracy                           0.96        54
   macro avg       0.96      0.96      0.96        54
weighted avg       0.96      0.96      0.96        54


Confusion Matrix:
[[18  0  0]
 [ 0 20  1]
 [ 0  1 14]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [21]:
student_model = tf.keras.Sequential([
    tf.keras.layers.Dense(
        32,
        activation='relu',
        input_shape=(num_features,)
    ),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

In [22]:
teacher_preds_soft = model.predict(X_train, verbose=0)

In [23]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

y_train_combined = np.concatenate(
    [y_train_cat, teacher_preds_soft],
    axis=1
)

def distillation_loss(y_true_combined, y_pred):

    y_true_hard = y_true_combined[:, :3]
    y_true_soft = y_true_combined[:, 3:]

    hard_loss = tf.keras.losses.categorical_crossentropy(
        y_true_hard,
        y_pred
    )

    soft_loss = tf.keras.losses.categorical_crossentropy(
        y_true_soft,
        y_pred
    )

    alpha = 0.5

    return alpha * hard_loss + (1 - alpha) * soft_loss

In [24]:
student_model.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

history_student = student_model.fit(
    X_train,
    y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 0s 6ms/step - loss: 1.0263 - accuracy: 0.4848 - val_loss: 0.9583 - val_accuracy: 0.6000
Epoch 2/10
13/13 [==============================] - 0s 1ms/step - loss: 0.8900 - accuracy: 0.7071 - val_loss: 0.8394 - val_accuracy: 0.8000
Epoch 3/10
13/13 [==============================] - 0s 1ms/step - loss: 0.7687 - accuracy: 0.8081 - val_loss: 0.7334 - val_accuracy: 0.8400
Epoch 4/10
13/13 [==============================] - 0s 1ms/step - loss: 0.6570 - accuracy: 0.8687 - val_loss: 0.6330 - val_accuracy: 0.9200
Epoch 5/10
13/13 [==============================] - 0s 1ms/step - loss: 0.5554 - accuracy: 0.9192 - val_loss: 0.5362 - val_accuracy: 0.9200
Epoch 6/10
13/13 [==============================] - 0s 1ms/step - loss: 0.4583 - accuracy: 0.9495 - val_loss: 0.4493 - val_accuracy: 0.9200
Epoch 7/10
13/13 [==============================] - 0s 1ms/step - loss: 0.3718 - accuracy: 0.9596 - val_loss: 0.3749 - val_accuracy: 0.9200
Epoch 8/10
13/13 [==

In [25]:
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
student_tflite_model = converter.convert()

with open("model_kd.tflite", "wb") as f:
    f.write(student_tflite_model)

kd_size_kb = file_size_kb("model_kd.tflite")

print(f"Knowledge Distillation model size: {kd_size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmpq_hngdg4/assets


INFO:tensorflow:Assets written to: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmpq_hngdg4/assets


Knowledge Distillation model size: 6.10 KB


2026-05-19 13:26:27.498063: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-19 13:26:27.498072: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-19 13:26:27.498169: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmpq_hngdg4
2026-05-19 13:26:27.498554: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 13:26:27.498557: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmpq_hngdg4
2026-05-19 13:26:27.499572: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-19 13:26:27.515597: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmpq_hngdg4
2026-05-

In [26]:
y_pred_probs = student_model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

y_true = np.argmax(y_test_cat, axis=1)

student_acc = np.mean(y_pred == y_true)

print(f"Student model accuracy: {student_acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Student model accuracy: 0.9630

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.90      0.95        21
           2       0.88      1.00      0.94        15

    accuracy                           0.96        54
   macro avg       0.96      0.97      0.96        54
weighted avg       0.97      0.96      0.96        54


Confusion Matrix:
[[18  0  0]
 [ 0 19  2]
 [ 0  0 15]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [27]:
print("Previous model comparison:")
print(f"Baseline float32 size: {base_size_kb:.2f} KB, accuracy: {test_acc:.4f}")
print(f"INT8 size: {int8_size:.2f} KB, accuracy: {int8_acc:.4f}")
print(f"Float16 size: {float16_size:.2f} KB, accuracy: {float16_acc:.4f}")
print(f"Dynamic size: {dynamic_size:.2f} KB, accuracy: {dynamic_acc:.4f}")
print(f"Pruned size: {pruned_size_kb:.2f} KB")
print(f"KD student size: {kd_size_kb:.2f} KB, accuracy: {student_acc:.4f}")

Previous model comparison:
Baseline float32 size: 14.07 KB, accuracy: 0.9630
INT8 size: 5.74 KB, accuracy: 0.9815
Float16 size: 8.95 KB, accuracy: 0.9815
Dynamic size: 8.17 KB, accuracy: 0.9815
Pruned size: 14.14 KB
KD student size: 6.10 KB, accuracy: 0.9630


In [28]:
# Strategy: apply full INT8 quantization to the smaller knowledge-distilled student model.

def student_representative_data_gen():
    for i in range(min(100, len(X_train))):
        yield [X_train[i:i + 1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = student_representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

kd_int8_tflite_model = converter.convert()

with open("model_kd_int8.tflite", "wb") as f:
    f.write(kd_int8_tflite_model)

kd_int8_size_kb = file_size_kb("model_kd_int8.tflite")

print(f"KD + INT8 TFLite model size: {kd_int8_size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmpvm3qu2f6/assets


INFO:tensorflow:Assets written to: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmpvm3qu2f6/assets


KD + INT8 TFLite model size: 3.62 KB


/Users/danie/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-19 13:26:27.764551: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-19 13:26:27.764562: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-19 13:26:27.764677: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmpvm3qu2f6
2026-05-19 13:26:27.765033: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-19 13:26:27.765037: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/4s/mlmz_j9d51l_mmhcg09sfpkw0000gn/T/tmpvm3qu2f6
2026-05-19 13:26:27.766023: I tensorflow/cc/saved_model/loader.cc:233] Re

In [29]:
# Evaluate KD + INT8 TFLite model.

interpreter = tf.lite.Interpreter(model_path="model_kd_int8.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_index = input_details[0]["index"]
output_index = output_details[0]["index"]

input_scale, input_zero_point = input_details[0]["quantization"]
output_scale, output_zero_point = output_details[0]["quantization"]

y_pred = []

for i in range(len(X_test)):
    input_data = X_test[i:i + 1].astype(np.float32)

    input_data = input_data / input_scale + input_zero_point
    input_data = np.round(input_data).astype(input_details[0]["dtype"])

    interpreter.set_tensor(input_index, input_data)
    interpreter.invoke()

    output_data = interpreter.get_tensor(output_index)

    output_data = output_scale * (
        output_data.astype(np.float32) - output_zero_point
    )

    y_pred.append(np.argmax(output_data, axis=1)[0])

y_pred = np.array(y_pred)
y_true = np.argmax(y_test_cat, axis=1)

kd_int8_acc = np.mean(y_pred == y_true)

print(f"KD + INT8 accuracy: {kd_int8_acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

KD + INT8 accuracy: 0.9630

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.90      0.95        21
           2       0.88      1.00      0.94        15

    accuracy                           0.96        54
   macro avg       0.96      0.97      0.96        54
weighted avg       0.97      0.96      0.96        54


Confusion Matrix:
[[18  0  0]
 [ 0 19  2]
 [ 0  0 15]]


In [30]:
final_comparison = pd.DataFrame({
    "Model": [
        "Baseline Float32",
        "Dynamic Quantized",
        "Full INT8 Quantized",
        "Float16 Quantized",
        "Pruned",
        "Knowledge Distilled Student",
        "KD Student + INT8"
    ],
    "Size (KB)": [
        base_size_kb,
        dynamic_size,
        int8_size,
        float16_size,
        pruned_size_kb,
        kd_size_kb,
        kd_int8_size_kb
    ],
    "Accuracy": [
        test_acc,
        dynamic_acc,
        int8_acc,
        float16_acc,
        test_acc,
        student_acc,
        kd_int8_acc
    ]
})

final_comparison

,Model,Size (KB),Accuracy
0,Baseline Float32,14.070312,0.962963
1,Dynamic Quantized,8.171875,0.981481
2,Full INT8 Quantized,5.742188,0.981481
3,Float16 Quantized,8.945312,0.981481
4,Pruned,14.140625,0.962963
5,Knowledge Distilled Student,6.101562,0.962963
6,KD Student + INT8,3.617188,0.962963


Conclusion:
The smallest model from the earlier parts was identified by comparing the saved TFLite
file sizes from quantization, pruning, and knowledge distillation. To reduce the model
size further, I combined knowledge distillation with full integer INT8 quantization.

Knowledge distillation reduced the number of parameters by training a smaller student
network, while INT8 quantization reduced the precision of the model weights and
activations from 32-bit floating point values to 8-bit integers. This combination
produced a smaller TFLite model than using the original baseline model alone.

The main factor that reduced size was the smaller student architecture. INT8
quantization then further compressed that already smaller model. The final accuracy
should be compared with the baseline and earlier compressed models to decide whether
the size reduction caused a significant performance loss.

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
